Offline teaching language view. Code, existing outputs and configuration retain the accepted reader identity. Fixed filing facts and reference figures are historical teaching inputs; new constructed calculations are not market-strategy reproduction.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). This stage delivers language views; portable dependencies and execution are checked in the later isolated cold start.

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch34_market_microstructure_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# Chapter 34: a one-venue static order-book sweep

The book and all values are constructed. The calculator consumes visible depth at a static snapshot, not an exchange matching engine, market feed or historical execution record. Prices use Decimal USD/share and quantities use integer shares; there is no download or random simulation.

In [ ]:
from decimal import Decimal
from pathlib import Path
from quant_course.microstructure import Level, OrderBook, sweep_displayed_book

book = OrderBook(
    bids=(Level(Decimal("100.00"), 300), Level(Decimal("99.98"), 400)),
    asks=(Level(Decimal("100.02"), 200), Level(Decimal("100.04"), 300)),
)
mid = book.midpoint()
spread = book.quoted_spread()
buy = sweep_displayed_book(book, "buy", 350)
sell = sweep_displayed_book(book, "sell", 350)

print({"best_bid": str(book.bids[0].price), "best_ask": str(book.asks[0].price),
       "mid": str(mid), "spread": str(spread)})
print("buy fills:", [(str(x.price), x.size) for x in buy.fills],
      "notional:", buy.notional, "VWAP:", buy.vwap)
print("hypothetical sell fills:", [(str(x.price), x.size) for x in sell.fills],
      "proceeds:", sell.notional, "VWAP:", sell.vwap)

In [ ]:
assert mid == Decimal("100.01")
assert spread == Decimal("0.02")
assert [(x.price, x.size) for x in buy.fills] == [
    (Decimal("100.02"), 200), (Decimal("100.04"), 150)
]
assert buy.notional == Decimal("35010.00")
assert buy.vwap == Decimal("35010") / 350
assert [(x.price, x.size) for x in sell.fills] == [
    (Decimal("100.00"), 300), (Decimal("99.98"), 50)
]
assert sell.notional == Decimal("34999.00")
arrival_mid_mark_pnl = Decimal(buy.filled) * mid - buy.notional
round_trip_pnl = sell.notional - buy.notional
assert arrival_mid_mark_pnl == Decimal("-6.50")
assert round_trip_pnl == Decimal("-11.00")
print({"arrival_mid_mark_pnl_usd": arrival_mid_mark_pnl,
       "hypothetical_round_trip_pnl_usd": round_trip_pnl})

## Interpreting the result

Mid USD 100.01 is a reference mark rather than a promised fill. Buying 350 shares consumes 200 at USD 100.02 and 150 at USD 100.04, costing USD 35,010. A hypothetical immediate sale into the same frozen book gives USD 34,999: round-trip P&L is −USD 11, compared with arrival-mid mark P&L of −USD 6.50. Fees, latency, queues, hidden depth, cancellations, routing and market response are omitted. The thin-book example does not invent a price for 250 uncovered shares.

In [ ]:
thin_book = OrderBook(
    bids=(),
    asks=(Level(Decimal("100.02"), 100),),
)
partial = sweep_displayed_book(thin_book, "buy", 350)
empty = sweep_displayed_book(OrderBook(bids=(), asks=()), "buy", 20)
assert (partial.filled, partial.remaining, partial.vwap) == (
    100, 250, Decimal("100.02")
)
assert (empty.filled, empty.remaining, empty.vwap) == (0, 20, None)
print({"partial_fill": partial.filled, "uncovered": partial.remaining,
       "no_fill_vwap": empty.vwap})

## Clock contract and OHLCV

Constructed UTC fields distinguish event, availability, ingestion, decision, send, venue receipt and fill. Daily OHLCV cannot reconstruct that event sequence or prove a touched limit filled.

In [ ]:
from datetime import datetime, timezone

clock_example = {
    "event_time": datetime.fromisoformat("2026-10-08T14:30:00+00:00"),
    "available_at": datetime.fromisoformat("2026-10-08T14:30:00.050000+00:00"),
    "ingested_at": datetime.fromisoformat("2026-10-08T14:30:00.080000+00:00"),
    "decision_time": datetime.fromisoformat("2026-10-08T14:30:00.100000+00:00"),
    "order_send_time": datetime.fromisoformat("2026-10-08T14:30:00.110000+00:00"),
    "order_receive_time": datetime.fromisoformat("2026-10-08T14:30:00.120000+00:00"),
    "fill_time": datetime.fromisoformat("2026-10-08T14:30:00.125000+00:00"),
}
assert all(value.tzinfo == timezone.utc for value in clock_example.values())
assert clock_example["available_at"] <= clock_example["decision_time"]
assert clock_example["decision_time"] < clock_example["order_send_time"]
assert clock_example["order_send_time"] <= clock_example["order_receive_time"]
assert clock_example["order_receive_time"] <= clock_example["fill_time"]
print({key: value.isoformat() for key, value in clock_example.items()})

In [ ]:
import runpy
from IPython.display import Image, display

root = Path.cwd()
script = runpy.run_path(str(root / "code/ch34_market_microstructure.py"))
figure_path = RUN_DIR / "figures/ch34_order_book_sweep.png"
script["plot_case"](book, figure_path)
display(Image(filename=str(figure_path)))

## Questions and figure

1. Why does arrival-mid mark P&L differ from hypothetical round-trip P&L?
2. If the best ask has 100 shares against a request for 350, what can be calculated and what needs extra rules?
3. Which event fields help reconstruct order state, and which cannot be inferred from OHLCV?

The new plot shows visible shares versus USD/share and a USD round-trip decomposition. It is saved under RUN_DIR/figures.